<a href="https://colab.research.google.com/github/zarazakaryan-byte/automation-portfolio/blob/main/excel_vs_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 7 — Excel vs Python: Same Mess, Two Tools

**Goal:** clean the Day 3 messy sales data using pandas, side by side with what you already did in Excel and Power Query.

**The point isn't to learn pandas.** You already know it from Lesson 1. The point is to **build judgment about which tool to reach for** in real work.

**Structure:** for each cleaning step, you'll see:
- 🟢 What you did in **Excel**
- 🟣 What you did in **Power Query**
- 🟠 What pandas does (with code you'll run)

By the end you'll have one summary table: when each tool wins.

---

## Setup

Run the cell below to install dependencies (pandas is already in Colab; we just need to verify) and download the messy dataset directly from the URL.


In [1]:
import pandas as pd
import numpy as np

# Check versions
print(f'pandas version: {pd.__version__}')
print(f'numpy version:  {np.__version__}')

# Display options — more readable in Colab
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)


pandas version: 2.2.2
numpy version:  2.0.2


## Step 0 — Load the data

We're going to recreate the same 49-row messy dataset Claude generated for Day 3. Since you don't have that file in Colab, we'll just generate it again in code. This is the exact same data you cleaned in Excel.

**Tool comparison for this step:**
- 🟢 Excel: File → Open the .xlsx file
- 🟣 Power Query: Data → From Sheet (opens editor)
- 🟠 Pandas: `pd.read_csv('file.csv')` or `pd.read_excel('file.xlsx')`

Three different mechanics, same outcome: data is now in memory, ready to clean.


In [2]:
# Recreate the Day 3 messy dataset
import random
from datetime import datetime, timedelta

random.seed(7)

cities_clean = ['Yerevan', 'Tbilisi', 'Baku', 'Moscow', 'Istanbul', 'Dubai']
products_clean = ['Wireless Mouse', 'Mechanical Keyboard', 'USB-C Cable', 'Laptop Stand', 'Notebook A5', 'Desk Lamp']
prices = {'Wireless Mouse': 25, 'Mechanical Keyboard': 89.99, 'USB-C Cable': 12.50,
          'Laptop Stand': 45, 'Notebook A5': 8.75, 'Desk Lamp': 38}
first_names = ['Anna', 'David', 'Maria', 'Levon', 'Nina', 'Aram', 'Lilit', 'Sergey', 'Elena', 'Karen']
last_names = ['Petrosyan', 'Hakobyan', 'Sargsyan', 'Ivanov', 'Grigoryan', 'Smith', 'Mkrtchyan']

rows = []
start = datetime(2024, 1, 1)
for i in range(1, 46):
    order_id = f'ORD{1000+i}'
    date = start + timedelta(days=random.randint(0, 364))
    name = f'{random.choice(first_names)} {random.choice(last_names)}'
    city = random.choice(cities_clean)
    product = random.choice(products_clean)
    qty = random.randint(1, 10)
    price = prices[product]
    email = f"{name.lower().replace(' ', '.')}@example.com"
    rows.append([order_id, date.strftime('%Y-%m-%d'), name, city, product, qty, price, email])

# Inject the same mess
rows[2][3] = 'yerevan'
rows[7][3] = 'YEREVAN '
rows[11][3] = ' Tbilisi'
rows[15][3] = 'moscow'
rows[19][3] = 'ISTANBUL'
rows[23][3] = 'dubai '
rows[27][3] = '  Baku'
rows[4][4] = 'wireless mouse'
rows[9][4] = 'USB C Cable'
rows[14][4] = 'Mech. Keyboard'
rows[22][4] = 'DESK LAMP'
rows[33][4] = 'notebook a5'
rows[5][2] = None
rows[8][3] = None
rows[13][6] = None
rows[20][5] = None
rows[30][7] = ''
rows[40][2] = None
rows.append(list(rows[3]))
rows.append(list(rows[10]))
rows.append(list(rows[25]))
near_dup = list(rows[6])
near_dup[3] = (near_dup[3].upper() if near_dup[3] else 'MOSCOW')
rows.append(near_dup)
rows[12][5] = 999
rows[35][5] = -3
rows[17][1] = '15/03/2024'
rows[28][1] = '2024.04.22'
rows[37][1] = 'March 5, 2024'
rows[16][7] = 'not-an-email'
rows[31][7] = 'missing@dotcom'
random.shuffle(rows)

df = pd.DataFrame(rows, columns=['Order_ID', 'Order_Date', 'Customer_Name', 'City', 'Product', 'Quantity', 'Unit_Price', 'Email'])

print(f'Loaded {len(df)} rows of messy data')
df.head(10)


Loaded 49 rows of messy data


,Order_ID,Order_Date,Customer_Name,City,Product,Quantity,Unit_Price,Email
0,ORD1009,2024-02-22,Karen Grigoryan,None,Mechanical Keyboard,6.0,89.99,karen.grigoryan@example.com
1,ORD1014,2024-06-02,Elena Ivanov,Baku,Desk Lamp,8.0,NaN,elena.ivanov@example.com
2,ORD1005,2024-03-04,Levon Smith,Dubai,wireless mouse,1.0,8.75,levon.smith@example.com
3,ORD1010,2024-02-19,Elena Smith,Yerevan,USB C Cable,1.0,8.75,elena.smith@example.com
4,ORD1036,2024-01-14,David Mkrtchyan,Tbilisi,Notebook A5,-3.0,8.75,david.mkrtchyan@example.com
5,ORD1035,2024-10-17,Maria Grigoryan,Yerevan,USB-C Cable,10.0,12.50,maria.grigoryan@example.com
6,ORD1043,2024-09-22,Aram Hakobyan,Baku,Mechanical Keyboard,9.0,89.99,aram.hakobyan@example.com
7,ORD1033,2024-02-01,Levon Petrosyan,Tbilisi,Laptop Stand,3.0,45.00,levon.petrosyan@example.com
8,ORD1041,2024-01-12,None,Baku,Mechanical Keyboard,9.0,89.99,levon.grigoryan@example.com
9,ORD1044,2024-10-04,Elena Sargsyan,Dubai,Mechanical Keyboard,10.0,89.99,elena.sargsyan@example.com


## Step 1 — Audit the mess

Before cleaning, catalog what's wrong.

| Tool | What you do |
|---|---|
| 🟢 Excel | Click each column filter dropdown, scan distinct values, write notes |
| 🟣 Power Query | Same — visual scan via column filter |
| 🟠 Pandas | One line: `df.info()` + `df.describe()` + `df.isnull().sum()` |

**Pandas wins this step by a lot.** Three function calls give you a complete audit in under a second. In Excel it took you ~15 minutes.


In [3]:
# Audit 1: shape, types, and missing values overview
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Order_ID       49 non-null     object 
 1   Order_Date     49 non-null     object 
 2   Customer_Name  47 non-null     object 
 3   City           48 non-null     object 
 4   Product        49 non-null     object 
 5   Quantity       48 non-null     float64
 6   Unit_Price     48 non-null     float64
 7   Email          49 non-null     object 
dtypes: float64(2), object(6)
memory usage: 3.2+ KB


In [4]:
# Audit 2: missing values per column
df.isnull().sum()


,0
Order_ID,0
Order_Date,0
Customer_Name,2
City,1
Product,0
Quantity,1
Unit_Price,1
Email,0


In [5]:
# Audit 3: distinct values in text columns (spotting casing/spacing issues)
print('Cities:')
print(df['City'].value_counts())
print()
print('Products:')
print(df['Product'].value_counts())


Cities:
City
Dubai       11
Moscow       9
Yerevan      7
Tbilisi      6
Baku         4
Istanbul     3
  Baku       1
yerevan      1
 Tbilisi     1
ISTANBUL     1
moscow       1
YEREVAN      1
dubai        1
MOSCOW       1
Name: count, dtype: int64

Products:
Product
Mechanical Keyboard    10
Laptop Stand            9
Notebook A5             8
Wireless Mouse          8
Desk Lamp               6
USB-C Cable             3
wireless mouse          1
USB C Cable             1
Mech. Keyboard          1
DESK LAMP               1
notebook a5             1
Name: count, dtype: int64


In [6]:
# Audit 4: numeric summary (spots outliers immediately)
df[['Quantity', 'Unit_Price']].describe()


,Quantity,Unit_Price
count,48.000000,48.000000
mean,26.375000,40.492500
std,143.407858,30.162078
min,-3.000000,8.750000
25%,3.750000,12.500000
50%,6.500000,38.000000
75%,8.250000,45.000000
max,999.000000,89.990000


**Did you spot everything you spotted in Excel?**

Look at the City output — every casing variation (Yerevan, yerevan, YEREVAN, ' Tbilisi', etc.) appears as a separate row. In Excel you scrolled through a filter dropdown. Here it's one printout.

Look at `describe()` — `max=999, min=-3` for Quantity. Outliers caught instantly. In Excel you computed Q1/Q3/IQR with 4 formulas. Here, one line.

**The lesson:** for *audit*, pandas is dramatically faster. The trade-off is you have to know what to ask. Excel makes you see everything; pandas only shows what you ask for.


## Step 2 — Trim & standardize text

| Tool | Code/clicks |
|---|---|
| 🟢 Excel | `=PROPER(TRIM(D2))` in helper column, paste-as-values, delete helper |
| 🟣 Power Query | Select columns → Transform → Trim → Capitalize Each Word |
| 🟠 Pandas | `.str.strip().str.title()` |

Pandas is one line per column. **But notice — pandas does it in place** (well, returns a new Series). In Excel you needed the helper-column dance to avoid breaking formulas.


In [7]:
# Clean text columns: strip whitespace and title-case
text_cols = ['Customer_Name', 'City', 'Product']

for col in text_cols:
    df[col] = df[col].str.strip().str.title()

# Fix acronyms that .title() mangles (same as PROPER in Excel)
df['Product'] = df['Product'].replace({
    'Usb-C Cable': 'USB-C Cable',
    'Usb C Cable': 'USB-C Cable',
    'Mech. Keyboard': 'Mechanical Keyboard'
})

print('Cities after cleanup:')
print(df['City'].value_counts())
print()
print('Products after cleanup:')
print(df['Product'].value_counts())


Cities after cleanup:
City
Dubai       12
Moscow      11
Yerevan      9
Tbilisi      7
Baku         5
Istanbul     4
Name: count, dtype: int64

Products after cleanup:
Product
Mechanical Keyboard    11
Wireless Mouse          9
Laptop Stand            9
Notebook A5             9
Desk Lamp               7
USB-C Cable             4
Name: count, dtype: int64


**Same result as Excel/PQ** — Cities collapse to 6, Products to 6. But notice:

- **Excel:** 5+ clicks per column (helper column → formula → paste-as-values → delete helper → repeat)
- **Power Query:** 2 clicks per transform, but multi-column select makes it 1 step for all three columns
- **Pandas:** 1 loop, runs in milliseconds

**For repetitive transforms across many columns, pandas wins.** For occasional one-offs, Excel/PQ feel more discoverable.


## Step 3 — Remove duplicates

| Tool | How |
|---|---|
| 🟢 Excel | Data → Remove Duplicates → pick columns to check |
| 🟣 Power Query | Home → Remove Duplicates |
| 🟠 Pandas | `df.drop_duplicates(subset='Order_ID')` |

All three give you the choice of which columns to dedupe on. **In pandas you're forced to be explicit** (`subset=` parameter), which I'd argue is good — it prevents the accidental "check all columns" choice.


In [8]:
before = len(df)
df = df.drop_duplicates(subset='Order_ID', keep='first').reset_index(drop=True)
after = len(df)

print(f'Removed {before - after} duplicate rows. Now have {after} rows.')


Removed 4 duplicate rows. Now have 45 rows.


## Step 4 — Handle missing values

Four strategies (same as Day 3):

1. **Delete the row** → `df.dropna(subset='Quantity')`
2. **Fill with default** → `df['City'].fillna('Unknown')`
3. **Fill with calculated** → `df.merge(prices, on='Product')` then fillna
4. **Flag and keep** → add a `Has_Missing` boolean column

Pandas does all four in one or two lines each.


In [9]:
# Strategy 1: Drop rows with missing critical fields (Quantity)
before = len(df)
df = df.dropna(subset=['Quantity']).reset_index(drop=True)
print(f'Dropped {before - len(df)} row(s) missing Quantity')

# Strategy 2: Fill text columns with defaults
df['Customer_Name'] = df['Customer_Name'].fillna('Unknown Customer')
df['City'] = df['City'].fillna('Unknown')
df['Email'] = df['Email'].replace('', None).fillna('MISSING')

# Strategy 3: Fill missing Unit_Price by looking up the Product
# This is the pandas version of XLOOKUP from Day 3
price_lookup = {
    'Wireless Mouse': 25, 'Mechanical Keyboard': 89.99, 'USB-C Cable': 12.50,
    'Laptop Stand': 45, 'Notebook A5': 8.75, 'Desk Lamp': 38, 'Coffee Mug': 14.50
}
df['Unit_Price'] = df['Unit_Price'].fillna(df['Product'].map(price_lookup))

print(f'\nMissing values per column after cleanup:')
print(df.isnull().sum())


Dropped 1 row(s) missing Quantity

Missing values per column after cleanup:
Order_ID         0
Order_Date       0
Customer_Name    0
City             0
Product          0
Quantity         0
Unit_Price       0
Email            0
dtype: int64


**Look at the elegance of `df['Product'].map(price_lookup)`.**

In Excel you wrote: `=XLOOKUP(E2, Products!A:A, Products!D:D)`

In pandas: `df['Product'].map(price_lookup)` does the same lookup — match each Product to its price in the dictionary, return the matching price. One method, two arguments, done.

**This is the pattern you'll use constantly:** dictionaries + `.map()` = pandas' XLOOKUP.


## Step 5 — Fix date types

Remember the Day 3 trap? Dates stored as text. ISNUMBER returned FALSE. Text to Columns to fix it.

In pandas: `pd.to_datetime()` parses mixed formats automatically — *if* you give it the right hint.

**The killer feature:** pandas can handle the mixed formats (`15/03/2024`, `2024.04.22`, `March 5, 2024`) **in a single line** with `format='mixed'`. Excel forced you to find-and-replace each oddball first.


In [10]:
# Convert Order_Date from text to real datetime
df['Order_Date'] = pd.to_datetime(df['Order_Date'], format='mixed', dayfirst=False)

print('Order_Date dtype:', df['Order_Date'].dtype)
print()
print('First few dates after conversion:')
print(df['Order_Date'].head())
print()
print('Storage vs display: each date is stored as a number internally')
print('March 5, 2024 internal value:', pd.Timestamp('2024-03-05').value)


Order_Date dtype: datetime64[ns]

First few dates after conversion:
0   2024-02-22
1   2024-06-02
2   2024-03-04
3   2024-02-19
4   2024-01-14
Name: Order_Date, dtype: datetime64[ns]

Storage vs display: each date is stored as a number internally
March 5, 2024 internal value: 1709596800000000000


**Same "storage vs display" concept as Day 3.** Pandas stores datetime as nanoseconds since 1970 (a number); the print shows it formatted as `YYYY-MM-DD`. Same idea as Excel storing 45356 and displaying `05.03.24`.

Tool comparison:
- 🟢 Excel: Find & Replace 3 oddball formats → Text to Columns → YMD picker (3-5 minutes)
- 🟣 Power Query: Click type icon → Date (sometimes errors on weird formats)
- 🟠 Pandas: one line, `format='mixed'` handles everything

**Pandas wins decisively here.** Mixed-format date columns are notoriously painful in Excel; pandas handles them in 50 milliseconds.


## Step 6 — Outliers (the IQR + business rule combo)

Day 3 lesson: IQR catches **999** but not **-3**. Statistical detection ≠ business validation. Use both.

In Excel: 4 separate formulas (Q1, Q3, IQR, bounds) + visual inspection. In pandas: `describe()` gives you Q1 and Q3 for free, the rest is one line of math.


In [11]:
# Statistical outliers via IQR
q1 = df['Quantity'].quantile(0.25)
q3 = df['Quantity'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(f'Q1={q1}, Q3={q3}, IQR={iqr}, Bounds=[{lower}, {upper}]')

# Flag outliers (statistical OR business rule)
df['Was_Outlier'] = (df['Quantity'] > upper) | (df['Quantity'] < 1)

# Show flagged rows BEFORE we modify them — the 'metadata before mutation' lesson from Day 3
print('\nFlagged rows:')
print(df[df['Was_Outlier']][['Order_ID', 'Quantity', 'Was_Outlier']])


Q1=3.0, Q3=8.0, IQR=5.0, Bounds=[-4.5, 15.5]

Flagged rows:
   Order_ID  Quantity  Was_Outlier
4   ORD1036      -3.0         True
43  ORD1013     999.0         True


In [12]:
# Now cap the values (winsorize the high end, floor the low end)
df['Quantity_Original'] = df['Quantity']  # preserve original
df.loc[df['Quantity'] > upper, 'Quantity'] = int(upper)  # cap at upper
df.loc[df['Quantity'] < 1, 'Quantity'] = 1                # floor at 1

print('After cleaning:')
print(df[df['Was_Outlier']][['Order_ID', 'Quantity_Original', 'Quantity', 'Was_Outlier']])


After cleaning:
   Order_ID  Quantity_Original  Quantity  Was_Outlier
4   ORD1036               -3.0       1.0         True
43  ORD1013              999.0      15.0         True


**Notice the ordering** — exactly the Day 3 lesson: *capture metadata BEFORE mutating data*. The `Was_Outlier` column was set while the original 999 and -3 values were still in place. If we'd modified Quantity first, the flag would be wrong.

Pandas elegance here: `df.loc[condition, 'column'] = value` is the standard idiom for "update where this condition is true." You'll use this constantly.


## Step 7 — Email validation

Day 3 lesson: basic validation is `has @ AND has .`, but our placeholder `missing@unknown.com` passes that test (false positive). We need granular status, not a binary.

In Excel: nested `AND(ISNUMBER(SEARCH...)...)`. In pandas: regex via `.str.contains()`, or a function for status.


In [13]:
# Granular email status (the Day 3 senior-level approach)
def email_status(email):
    if pd.isna(email) or email == 'MISSING':
        return 'missing'
    if '@' not in email:
        return 'invalid_no_at'
    if '.' not in email.split('@')[-1]:
        return 'invalid_no_tld'
    if email == 'missing@unknown.com':
        return 'placeholder'
    return 'valid'

df['Email_Status'] = df['Email'].apply(email_status)

print(df['Email_Status'].value_counts())


Email_Status
valid             41
invalid_no_at      1
missing            1
invalid_no_tld     1
Name: count, dtype: int64


## Step 8 — Save & verify

Excel: save the .xlsx. Power Query: Close & Load. Pandas: `to_csv()` or `to_excel()`.


In [14]:
# Final check: full summary
print('Final dataset summary')
print('=' * 40)
print(f'Rows: {len(df)}')
print(f'Columns: {len(df.columns)}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Outliers flagged: {df["Was_Outlier"].sum()}')
print(f'Date range: {df["Order_Date"].min()} to {df["Order_Date"].max()}')
print()
df.head(10)


Final dataset summary
Rows: 44
Columns: 11
Missing values: 0
Outliers flagged: 2
Date range: 2024-01-07 00:00:00 to 2024-12-27 00:00:00



,Order_ID,Order_Date,Customer_Name,City,Product,Quantity,Unit_Price,Email,Was_Outlier,Quantity_Original,Email_Status
0,ORD1009,2024-02-22,Karen Grigoryan,Unknown,Mechanical Keyboard,6.0,89.99,karen.grigoryan@example.com,False,6.0,valid
1,ORD1014,2024-06-02,Elena Ivanov,Baku,Desk Lamp,8.0,38.00,elena.ivanov@example.com,False,8.0,valid
2,ORD1005,2024-03-04,Levon Smith,Dubai,Wireless Mouse,1.0,8.75,levon.smith@example.com,False,1.0,valid
3,ORD1010,2024-02-19,Elena Smith,Yerevan,USB-C Cable,1.0,8.75,elena.smith@example.com,False,1.0,valid
4,ORD1036,2024-01-14,David Mkrtchyan,Tbilisi,Notebook A5,1.0,8.75,david.mkrtchyan@example.com,True,-3.0,valid
5,ORD1035,2024-10-17,Maria Grigoryan,Yerevan,USB-C Cable,10.0,12.50,maria.grigoryan@example.com,False,10.0,valid
6,ORD1043,2024-09-22,Aram Hakobyan,Baku,Mechanical Keyboard,9.0,89.99,aram.hakobyan@example.com,False,9.0,valid
7,ORD1033,2024-02-01,Levon Petrosyan,Tbilisi,Laptop Stand,3.0,45.00,levon.petrosyan@example.com,False,3.0,valid
8,ORD1041,2024-01-12,Unknown Customer,Baku,Mechanical Keyboard,9.0,89.99,levon.grigoryan@example.com,False,9.0,valid
9,ORD1044,2024-10-04,Elena Sargsyan,Dubai,Mechanical Keyboard,10.0,89.99,elena.sargsyan@example.com,False,10.0,valid


In [15]:
# Save to CSV (download via Colab sidebar after running)
df.to_csv('day7-cleaned-data.csv', index=False)
print('Saved as day7-cleaned-data.csv')
print('To download: click the folder icon in left sidebar → find the file → download')


Saved as day7-cleaned-data.csv
To download: click the folder icon in left sidebar → find the file → download


## Honest verdict — when to use which tool

Now that you've done the exact same workflow in three tools, the trade-offs are visible. Fill this table in based on your own experience:

| Step | Easiest in | Fastest in | Best for *real* analyst work |
|---|---|---|---|
| Audit (info/describe) | 🟠 Pandas | 🟠 Pandas | 🟠 Pandas |
| Trim & standardize text | 🟣 PQ | 🟠 Pandas | 🟣 PQ for repeatable, 🟠 Pandas for ad-hoc |
| Deduplication | 🟢 Excel | 🟠 Pandas | Tie |
| Missing values | 🟢 Excel (visual) | 🟠 Pandas | 🟣 PQ (refreshable) |
| Date type fix | 🟠 Pandas (`format='mixed'`) | 🟠 Pandas | 🟠 Pandas — Excel's nemesis |
| Outliers (IQR + flag) | 🟠 Pandas (one line) | 🟠 Pandas | 🟠 Pandas |
| Validation logic | 🟠 Pandas (custom funcs) | 🟠 Pandas | 🟠 Pandas |
| **Refresh on new data** | ❌ | 🟣 **PQ** | 🟣 PQ for recurring, 🟠 Pandas for scripted |
| **Visual exploration** | 🟢 **Excel** | 🟢 Excel | 🟢 Excel for stakeholders |
| **Scaling to 10M+ rows** | ❌ | 🟠 Pandas | 🟠 Pandas (or SQL) |

### The decision tree I'd use in a real job

```
Dataset under 100K rows?         → Excel is fine for exploration
                                                                                    
Will this clean repeat monthly?  → Power Query (one-click refresh)
                                                                                    
Need full reproducibility,       → Pandas (version control, code review)
team collaboration, scripting?                                                       
                                                                                    
Stakeholder will use it?         → Excel (universal tool)
                                                                                    
Joining multiple large tables?   → SQL > pandas > PQ                                
```

### The pattern I want you to internalize

**Tools are not competing — they layer.** Real analyst pipelines often use *all three*:

1. Raw data ingested via Power Query (refreshable, scheduled)
2. Heavy transformations in pandas/SQL (complex logic, large scale)
3. Final dashboard rendered in Excel/Power BI (for stakeholders)

Knowing which tool to reach for in which moment is the actual skill. **You now have direct evidence in your hands** — same workflow, three tools — to make those calls confidently.

---

## Reflections — your turn

Save this notebook to your Drive (File → Save a copy in Drive). Add a final cell below and write your honest answers:

1. Which tool felt **fastest** for the cleaning workflow overall?
2. Which felt **most reliable** (lowest chance of silent bugs)?
3. Which felt **most enjoyable to use**?
4. If you were given a fresh messy dataset Monday morning at a new job, which tool would you reach for first, and why?

These answers go into your Day 7 notes.


## Where to go next

- Save this notebook to your Drive (File → Save a copy in Drive)
- Download `day7-cleaned-data.csv` if you want it
- Add this notebook link/file to your `data-analytics-roadmap/notes/` folder

**Week 2 starts Monday: SQL.**

You've now seen data cleaning in 3 tools. Next week we add a 4th — but SQL isn't just "another way to clean data." It's the language of databases, and 90%+ of analyst job postings list SQL as a core requirement. Get ready.


In [ ]:
df.describe()